In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.io import loadmat
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from math import log, sqrt
from sklearn.metrics import confusion_matrix
import seaborn as sns
from sklearn.utils import shuffle
import time
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import ShuffleSplit
from sklearn.preprocessing import label_binarize
from itertools import cycle
from sklearnex import patch_sklearn
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

In [ ]:
patch_sklearn()

In [ ]:
X=loadmat("data/PaviaU.mat")['paviaU']
Y=loadmat("data/PaviaU_gt.mat")['paviaU_gt']

X.shape,Y.shape

In [ ]:
def plot_band_boxplot(X,Y):
    plt.figure(figsize=(16, 6))
    df=pd.DataFrame(X.reshape(X.shape[0]*X.shape[1], -1))
    df.colmns= [i for i in range(1, df.shape[-1]+1)]
    df['class'] = Y.ravel()
    sns.boxplot(x=df["class"], y=df[0], width=0.3)
    plt.title('Box Plot', fontsize=16)
    plt.xlabel('Class', fontsize=14)
    plt.show()
    return df

In [ ]:
df=plot_band_boxplot(X,Y)

In [ ]:
def Denoise_dataset(dataset):
    Denoised_x= df[df['class']!=0].iloc[:, :-1].values
    Denoised_y = df[df['class']!=0].iloc[:, -1].values
    Denoised_df = pd.concat([pd.DataFrame(Denoised_x), pd.Series(Denoised_y)], axis=1)
    Denoised_df.columns = [f'band-{i}' for i in range(1, 1+X.shape[2])]+['class']
    return Denoised_df

In [ ]:
Denoised_df=Denoise_dataset(df)

In [ ]:
np.unique(Denoised_df['class'])

In [ ]:
import statsmodels.api as sm
def backward_elimination(data, target,significance_level = 0.05):
    features = data.columns.tolist()
    while(len(features)>0):
        features_with_constant = sm.add_constant(data[features])
        p_values = sm.OLS(target, features_with_constant).fit().pvalues[1:]
        max_p_value = p_values.max()
        if(max_p_value >= significance_level):
            excluded_feature = p_values.idxmax()
            features.remove(excluded_feature)
        else:
            break 
    return features

In [ ]:
forward_start_time = time.time()
X=Denoised_df.iloc[:, :-1]
y=Denoised_df.iloc[:, -1]
best_features=backward_elimination(X,y)
print('Time taken by backward Feature Selection is :'+str(int(time.time() - forward_start_time))+' seconds')

df= pd.concat([Denoised_df[best_features], y], axis=1)
df

In [ ]:
df.shape

In [ ]:
def split_dataset(Denoised_df, test_size=0.2, random_state=42):
    unique_classes = set(Denoised_df.iloc[:,-1])
    train_data = {class_label: {'X': None, 'y': None} for class_label in unique_classes}
    test_data = {class_label: {'X': None, 'y': None} for class_label in unique_classes}
    for class_label in unique_classes:
        class_indices = [i for i, label in enumerate(Denoised_df.iloc[:,-1]) if label == class_label]
        X_class, y_class = Denoised_df.iloc[class_indices,:-1], Denoised_df.iloc[class_indices,-1]
        X_train, X_test, y_train, y_test = train_test_split(X_class, y_class, test_size=test_size, random_state=random_state)
        train_data[class_label]['X'] = X_train
        train_data[class_label]['y'] = y_train
        test_data[class_label]['X'] = X_test
        test_data[class_label]['y'] = y_test
        
    dfs_to_concat_X = [train_data[i]['X'] for i in unique_classes]
    dfs_to_concat_Y = [train_data[i]['y'] for i in unique_classes]
    merged_train_df_X = pd.concat(dfs_to_concat_X, ignore_index=True)
    merged_train_df_Y=  pd.concat(dfs_to_concat_Y, ignore_index=True)
    train_data = pd.concat([merged_train_df_X, merged_train_df_Y], axis=1)
    dfs_to_concat_X= [test_data[i]['X'] for i in unique_classes]
    dfs_to_concat_Y= [test_data[i]['y'] for i in unique_classes]
    merged_train_df_X = pd.concat(dfs_to_concat_X, ignore_index=True)
    merged_train_df_Y=  pd.concat(dfs_to_concat_Y, ignore_index=True)
    test_data= pd.concat([merged_train_df_X, merged_train_df_Y], axis=1)
    
    train_data=shuffle(train_data, random_state=42)
    test_data=shuffle(test_data, random_state=42)
    
    x_train= train_data.iloc[:, :-1]
    x_test=  test_data.iloc[:,:-1]
    y_train= train_data.iloc[:,-1]
    y_test= test_data.iloc[:,-1]    
    return x_train,x_test,y_train,y_test

In [ ]:
x_train,x_test,y_train,y_test=split_dataset(df)

In [ ]:
x_train.shape,x_test.shape,y_train.shape,y_test.shape

In [ ]:
class PrincipalComponentAnalysis:
    def __init__(self, n_components):
        self.n_components = n_components
        self.components = None
        self.mean = None
        self.explained_variance_ratio_ = None #New parameters 
    
    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        X = X - self.mean
        cov = np.cov(X.T)
        eigenvalues, eigenvectors = np.linalg.eig(cov)
        eigenvectors = eigenvectors.T
        idxs = np.argsort(-eigenvalues)  
        eigenvalues = eigenvalues[idxs]
        eigenvectors = eigenvectors[idxs]
        self.components = eigenvectors[:self.n_components]
        total_variance = np.sum(eigenvalues)
        explained_variance = eigenvalues[:self.n_components]
        self.explained_variance_ratio_ = explained_variance / total_variance
    
    def transform(self, X):
        X = X - self.mean
        return np.dot(X, self.components.T)
    # New function
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

In [ ]:
class KNN:
    def __init__(self, n_neighbors=30, weights='distance', algorithm='auto', metric='euclidean'):
        self.trainData = None
        self.trainLabel = None
        self.testData = None
        self.testLabel = None
        self.predict = None
        self._accuracy=[]
        self.clf = KNeighborsClassifier(
            n_neighbors=n_neighbors, 
            weights=weights, 
            algorithm=algorithm, 
            metric=metric
        )

    def KNN_SCORE(self, a, b, trainData, trainLabel, testData, testLabel):
        self.trainData = trainData
        self.trainLabel = trainLabel
        self.testData = testData
        self.testLabel = testLabel
        best_acc = -1  
        best_dim = None  
        best_y_pred = None  
        explained_variance = []

        for i in range(a, b + 1):
            print(f"Evaluating for {i} components")
            pca = PrincipalComponentAnalysis(i)
            tmp_x_train = pca.fit_transform(trainData)  
            self.clf.fit(tmp_x_train, trainLabel)
            temp_x_test=pca.transform(testData)
            y_pred = self.clf.predict(pca.transform(testData))
            accuracy = self.predict_score(y_pred, testLabel)
            explained_variance.append(np.sum(pca.explained_variance_ratio_)) 
            self._accuracy.append(accuracy)

            
            if accuracy > best_acc:
                best_acc = accuracy
                best_dim = i
                best_y_pred = y_pred
                best_dataset= tmp_x_train
                best_testData=  temp_x_test
                
                             
        print(f"The highest accuracy is {best_acc}, achieved with {best_dim} components.")
        self.predict = best_y_pred
        self.trainData = best_dataset
        self.testData= best_testData
        

        # 绘制精度和解释方差随主成分数量的变化图
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1)
        plt.plot(range(a, b + 1), self._accuracy, marker='o')
        plt.title('Model Accuracy vs. Number of Components')
        plt.xlabel('Number of Components')
        plt.ylabel('Accuracy')
        plt.grid()

        plt.subplot(1, 2, 2)
        plt.plot(range(a, b + 1), explained_variance, marker='o', color='orange')
        plt.title('Explained Variance vs. Number of Components')
        plt.xlabel('Number of Components')
        plt.ylabel('Explained Variance')
        plt.grid()

        plt.tight_layout()
        plt.show()

    def learnParams(self):
        param_grid = {
            'n_neighbors': list(range(5, 70, 5)),
            'weights': ['uniform', 'distance'],
            'algorithm': ['auto', 'ball_tree', 'kd_tree'],
            'metric': ['euclidean', 'manhattan']
        }
        cv = ShuffleSplit(n_splits=5, test_size=0.1, random_state=20)
        grid = GridSearchCV(KNeighborsClassifier(), param_grid=param_grid, cv=cv, n_jobs=-1)
        grid.fit(self.trainData, self.trainLabel)
        self.clf.set_params(**grid.best_params_)
        print(f"Best parameters: {grid.best_params_}, with a score of {grid.best_score_:.2f}")
        return grid.best_params_

    def crossValidation(self):
        cv = ShuffleSplit(n_splits=10, test_size=0.1, random_state=20)
        scores = cross_val_score(self.clf, self.trainData, self.trainLabel, cv=cv, scoring='accuracy')
        print(f"Accuracy: {scores.mean():.2f} ± {scores.std():.2f}")

        plt.figure(figsize=(10, 5))
        plt.plot(range(len(scores)), scores, marker='o', linestyle='-', color='blue', label='Individual cross-validation scores')
        plt.axhline(y=scores.mean(), color='r', linestyle='--', label=f'Mean Accuracy: {scores.mean():.2f}')
        plt.axhline(y=scores.mean() + scores.std(), color='g', linestyle='--', label=f'Mean Accuracy + 1 Std Dev: {scores.mean() + scores.std():.2f}')
        plt.axhline(y=scores.mean() - scores.std(), color='y', linestyle='--', label=f'Mean Accuracy - 1 Std Dev: {scores.mean() - scores.std():.2f}')
        plt.xlabel('Iteration')
        plt.ylabel('Accuracy')
        plt.title('Cross-Validation Scores')
        plt.legend()
        plt.show()

    def predict_score(self, y_true, y_pred):
        correct = 0
        total = len(y_true)
        for t, p in zip(y_true, y_pred):
            if t == p:
                correct += 1
        return correct / total

    def plot_confusion_matrix(self):
        plt.figure(figsize=(10, 7))
        classes = ['Asphalt', 'Meadows', 'Gravel', 'Trees', 'Painted metal sheets',
                   'Bare Soil', 'Bitumen', 'Self-Blocking Bricks', 'Shadows']
        mat = confusion_matrix(self.predict, self.testLabel)
        df_cm = pd.DataFrame(mat, index=classes, columns=classes)
        sns.heatmap(df_cm, annot=True, fmt='d')
        plt.show()
        
    def classification_report(self):
        report = classification_report(self.testLabel, self.predict)
        print(report)
    
    

In [ ]:
# Record the start time
start_time = time.time()
clf=KNN()
score=clf.KNN_SCORE(1,50,x_train,y_train,x_test,y_test)
end_time = time.time()
runtime = end_time - start_time
print(f"Code execution time: {runtime} seconds")

In [ ]:
# Record the start time
start_time = time.time()
clf.plot_confusion_matrix()
end_time = time.time()
runtime = end_time - start_time
print(f"Code execution time: {runtime} seconds")

In [ ]:
# Record the start time
start_time = time.time()
clf.classification_report()
end_time = time.time()
runtime = end_time - start_time
print(f"Code execution time: {runtime} seconds")

In [ ]:
# Record the start time
start_time = time.time()
clf.crossValidation()
end_time = time.time()
runtime = end_time - start_time
print(f"Code execution time: {runtime} seconds")

In [ ]:
# Record the start time
start_time = time.time()
best_params_= clf.learnParams()
end_time = time.time()
runtime = end_time - start_time
print(f"Code execution time: {runtime} seconds")

In [ ]:
def plot_roc_curve(model,testData,testLabel):
        """
        绘制ROC曲线。
        """
        # 获取每个类别的概率得分
        y_score = model.predict_proba(testData)
        
        n_classes=len(np.unique(testData))
        # 二值化标签（one-hot encoding）
        y_test_bin = label_binarize(testLabel, classes=np.arange(1, n_classes+1))
        n_classes=9
    

        # 计算每个类别的ROC曲线和AUC
        fpr = dict()
        tpr = dict()
        roc_auc = dict()
        for i in range(n_classes):
            fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])

        # 绘制所有ROC曲线
        plt.figure(figsize=(10, 10))
        colors = cycle(['aqua', 'darkorange', 'cornflowerblue', 'green', 'red', 'purple', 'pink', 'yellow', 'grey'])
        for i, color in zip(range(n_classes), colors):
            plt.plot(fpr[i], tpr[i], color=color, lw=2,
                     label=f'ROC curve of class {i} (area = {roc_auc[i]:.2f})')

        plt.plot([0, 1], [0, 1], 'k--', lw=2)
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('Receiver Operating Characteristic')
        plt.legend(loc="lower right")
        plt.show()
        


In [ ]:
# Record the start time
start_time = time.time()

KNN = KNeighborsClassifier(
           **best_params_
        )
KNN.fit(clf.trainData, clf.trainLabel)
plot_roc_curve(KNN,clf.testData,clf.testLabel)

end_time = time.time()
runtime = end_time - start_time
print(f"Code execution time: {runtime} seconds")